# Early stopping

_Deep Learning_

**Stop training the moment the validation error stops improving.**

Train too long and the network starts memorizing the training set. That is overfitting.

     Early stopping watches a separate validation set (data the model isn't trained on).

---

This notebook is a practice scaffold for the **Early stopping** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import copy

model = nn.Linear(10, 1)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
Xtr, ytr = torch.randn(200, 10), torch.randn(200, 1)
Xval, yval = torch.randn(50, 10), torch.randn(50, 1)

best_loss, best_state, patience, wait = float("inf"), None, 5, 0
for epoch in range(100):
    opt.zero_grad(); loss_fn(model(Xtr), ytr).backward(); opt.step()
    with torch.no_grad():
        val = loss_fn(model(Xval), yval).item()   # check validation
    if val < best_loss:
        best_loss, best_state, wait = val, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1
        if wait >= patience:                       # no improvement for 5 epochs
            print("stopped at epoch", epoch, "best val", round(best_loss, 4))
            break
model.load_state_dict(best_state)                  # restore best weights

## Visualize the data & results

_Training on real tumor data, when does validation loss bottom out and start rising?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss

bc = load_breast_cancer()
X = StandardScaler().fit_transform(bc.data)
Xtr, Xte, ytr, yte = train_test_split(X, bc.target, test_size=0.3,
                                      random_state=0, stratify=bc.target)

m = MLPClassifier(hidden_layer_sizes=(64, 64), solver="adam", max_iter=1,
                  warm_start=True, random_state=0, alpha=1e-5,
                  learning_rate_init=0.003)
tr, va = [], []
for e in range(40):
    m.partial_fit(Xtr, ytr, classes=[0, 1]) if e == 0 else m.partial_fit(Xtr, ytr)
    tr.append(log_loss(ytr, m.predict_proba(Xtr), labels=[0, 1]))
    va.append(log_loss(yte, m.predict_proba(Xte), labels=[0, 1]))

stop = int(np.argmin(va))               # real early-stopping epoch = 12
plt.plot(tr, color="#4ea1ff", label="train loss")
plt.plot(va, color="#ffb454", label="validation loss")
plt.axvline(stop, color="gray", linestyle="--", label="early stop")
plt.title("Real breast-cancer training: validation bottoms out at epoch 12")
plt.xlabel("epoch"); plt.ylabel("log loss")
plt.legend()
plt.show()